## Importing Libraries and functions

In [0]:
from pyspark.sql.functions import *
from src.logic.transformation_functions import *

## Transformations for orders table
Clean data: 
- Null order_id or customer_id, 
- Invalid order_date, 
- Duplicate order records, 
- Filter only valid order statuses (e.g., Completed, Shipped), 
- Standardize date formats 

In [0]:
# Reading orders table 
order_df = spark.read.format("delta").load("/Volumes/ecommerce_project/bronze_ecom/orders")

# display orders data
order_df.limit(10).display()


In [0]:
# standardizing the order_dates
clean_orders = standard_date(order_df, "order_date", "order_date")

# performing other transformations
clean_orders = clean_orders.filter(col("order_id").isNotNull())\
                        .filter(col("customer_id").isNotNull())\
                        .filter(col("order_date").isNotNull())\
                        .dropDuplicates(["order_id"])\
                        .filter(col("order_status").isin(["Completed","Shipped"]))\
                        .select('order_id','customer_id','order_date','order_status')

display(clean_orders.limit(10))


In [0]:
# writing the data
save_df_to_delta(df=clean_orders, mode="overwrite", path="ecommerce_project.silver_ecom.orders_silver")

## Transformation Customers data

Clean data: 
- Null customer names, 
- Invalid city values, 
- Duplicate customer records, 
- Standardize customer names, 
- Validate signup_date

In [0]:
# Reading customers data
customer_df = spark.read.format("delta").load("/Volumes/ecommerce_project/bronze_ecom/customers")

display(customer_df.limit(10))

In [0]:
# Transforming city names
customer_silver = transformed_city(customer_df, "city")

# Standardizing the signup date
customer_silver = standard_date(customer_silver, "signup_date", "signup_date")

# Performing other transformations
customer_silver = customer_silver.filter(col("customer_name").isNotNull())\
                             .dropDuplicates(["customer_id"])\
                             .withColumn("customer_name", string_title_format("customer_name"))\
                             .select("customer_id", "customer_name","city", "signup_date")



In [0]:
customer_silver.display()

In [0]:
# writing the data
save_df_to_delta(df=customer_silver, mode="overwrite", path="ecommerce_project.silver_ecom.customers_silver")

## Transformation of order_items data
Clean data: 
- Null order_id, 
- Quantity ≤ 0, 
- Price ≤ 0

**Calculate:** 

    Total_price = quantity * price 

**Derived Silver Metrics:**

- Order-level total amount. 
- Number of items per order. 

In [0]:
# Reading orders table 
items_df = spark.read.format("delta").load("/Volumes/ecommerce_project/bronze_ecom/order_items")

# display orders data
items_df.limit(10).display()

In [0]:

# Filtering data with not null order id values
order_items_silver = items_df.filter(col("order_id").isNotNull())

# Performing other transformations
order_items_silver_clean = order_items_silver.filter(col("quantity") > 0)\
                                    .filter(col("price").cast("double") > 0)\
                                    .withColumn("total_price", 
                                                round(col("quantity") * col("price").cast("double"), 2)
                                               )\
                                    .select("order_item_id","order_id", "product_name", "quantity", "price", "total_price")


# saving as delta table
save_df_to_delta(order_items_silver_clean, "overwrite", "ecommerce_project.silver_ecom.order_items_silver")

In [0]:
order_items_silver_clean.limit(10).display()

In [0]:
# Getting rows with negative quantity and price values
quarantine_order_items = order_items_silver.filter(
                                (col("quantity") <= 0) |
                                (col("price").cast("double") <= 0)
                            )\
                            .select("order_item_id","order_id", "product_name", "quantity", "price")


# saving the quarintine order_items data
save_df_to_delta(quarantine_order_items, "overwrite", "ecommerce_project.silver_ecom.quarantine_order_items")

In [0]:
# Performing Order level Aggregations
orders_enriched_silver = order_items_silver_clean.groupBy("order_id")\
                                .agg(
                                    round(sum("total_price"), 2).alias("order_level_total_amt"),
                                    count("*").alias("items_per_order")
                                )

# saving orders_enriched_silver as delta table
save_df_to_delta(orders_enriched_silver, "overwrite", "ecommerce_project.silver_ecom.orders_enriched_silver")